<a href="https://colab.research.google.com/github/2403a54127-lab/Natural-language-processing/blob/main/NLP_Lab_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer

Sample Dataset

In [2]:
texts = [
    "this movie is amazing",
    "i hate this film",
    "best movie ever",
    "worst acting ever",
    "i love this",
    "not good at all"
]

labels = [1, 0, 1, 0, 1, 0]  # 1 = positive, 0 = negative


Text Preprocessing

In [3]:
def tokenize(text):
    return text.lower().split()

tokenized_texts = [tokenize(t) for t in texts]


Build Vocabulary

In [4]:
vocab = {}
index = 1

for sentence in tokenized_texts:
    for word in sentence:
        if word not in vocab:
            vocab[word] = index
            index += 1

vocab_size = len(vocab) + 1  # +1 for padding


Encode & Pad Sequences

In [5]:
max_len = 6

def encode(sentence):
    encoded = [vocab[word] for word in sentence]
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    return encoded[:max_len]

encoded_texts = [encode(s) for s in tokenized_texts]

Convert text to BoW

In [6]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts).toarray()
y = labels



Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    encoded_texts, labels, test_size=0.2, random_state=42
)


Create Dataset class

In [8]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

Build 1D CNN Model

In [9]:
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Calculate flattened size correctly based on input_size (max_len)
        conv_output_length = input_size - kernel_size + 1
        # Assuming default stride=pool_size for MaxPool1d
        pool_output_length = (conv_output_length - pool_size) // pool_size + 1
        flattened_size = kernels * pool_output_length

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 neurons (you can change)

        # Output layer
        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer
        return self.sigmoid(x)          # Apply sigmoid for binary classification

Initialize Model

In [10]:
model = CNNModel(input_size=max_len)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


Training

In [11]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # Cast X_batch to float before passing to the model
        output = model(X_batch.float()).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7112
Epoch 2, Loss: 0.5805
Epoch 3, Loss: 0.5882
Epoch 4, Loss: 0.5544
Epoch 5, Loss: 0.5222
Epoch 6, Loss: 0.6575
Epoch 7, Loss: 0.4327
Epoch 8, Loss: 0.3754
Epoch 9, Loss: 0.3356
Epoch 10, Loss: 0.5127



Evaluation

In [12]:

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # Cast X_batch to float before passing to the model
        outputs = model(X_batch.float()).squeeze()
        preds = (outputs > 0.5).int()

        y_true.extend(y_batch.tolist())
        y_pred.extend(preds.tolist())
y_actual = [int(y) for y in y_true]
print(y_pred)

[0, 0]


Metrics

In [13]:
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Confusion Matrix:
[[1 0]
 [1 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Classification Report

In [14]:
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_actual, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
